<a href="https://colab.research.google.com/github/ryancookd/RNA-3D-Folding/blob/initial-commit/RNA_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

stanford_rna_3d_folding_2_path = kagglehub.competition_download('stanford-rna-3d-folding-2')

print('Data source import complete.')


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf

DATA_DIR = "/kaggle/input/stanford-rna-3d-folding-2"
MSA_DIR = os.path.join(DATA_DIR, "MSA")

SEED = 42
rng = np.random.default_rng(SEED)

# CPU-friendly knobs
MAX_LEN   = 512
N_TARGETS = 300
EPOCHS    = 5
BATCH     = 16

print("DATA_DIR:", DATA_DIR)
print("Example MSA:", glob.glob(os.path.join(MSA_DIR, "*.fasta"))[:3])
print("MAX_LEN:", MAX_LEN, "N_TARGETS:", N_TARGETS, "EPOCHS:", EPOCHS, "BATCH:", BATCH)

In [ ]:
BASES = ["A", "C", "G", "U"]

def read_fasta_seqs(path):
    seqs = []
    with open(path, "r") as f:
        cur = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur:
                    seqs.append("".join(cur).upper())
                    cur = []
            else:
                cur.append(line.strip())
        if cur:
            seqs.append("".join(cur).upper())
    return seqs

def msa_features_for_target(target_id, target_seq):
    """
    Returns feats: (L, 6) float32 where L=len(target_seq)
    columns: [freq_A, freq_C, freq_G, freq_U, gap_rate, conservation]
    """
    path = os.path.join(MSA_DIR, f"{target_id}.MSA.fasta")
    if not os.path.exists(path):
        return None

    seqs = read_fasta_seqs(path)
    if len(seqs) == 0:
        return None

    aln0 = seqs[0]
    ncol = len(aln0)
    if any(len(s) != ncol for s in seqs):
        # malformed alignment
        return None

    # counts per column across sequences
    # consider '-' and '.' as gaps
    counts = {b: np.zeros(ncol, dtype=np.int32) for b in BASES}
    gap = np.zeros(ncol, dtype=np.int32)

    for s in seqs:
        for i, ch in enumerate(s):
            if ch in counts:
                counts[ch][i] += 1
            elif ch in ("-", "."):
                gap[i] += 1
            else:
                # treat other symbols as gap-ish/unknown
                gap[i] += 1

    nseq = len(seqs)
    gap_rate = gap / max(nseq, 1)

    # compute per-column base freqs excluding gaps (for entropy)
    base_sum = sum(counts[b] for b in BASES)
    denom = np.maximum(base_sum, 1)  # avoid divide by zero
    freqs = np.stack([counts[b] / denom for b in BASES], axis=1).astype(np.float32)  # (ncol,4)

    # entropy over A/C/G/U (ignore gaps)
    eps = 1e-8
    ent = -np.sum(freqs * np.log(freqs + eps), axis=1)  # (ncol,)
    ent_norm = ent / np.log(4.0)
    conservation = (1.0 - ent_norm).astype(np.float32)  # (ncol,)

    # map alignment columns -> ungapped positions using aln0
    feats = np.zeros((len(target_seq), 6), dtype=np.float32)
    pos = 0
    for col in range(ncol):
        ch0 = aln0[col]
        if ch0 in ("-", "."):
            continue
        if pos >= len(target_seq):
            break
        feats[pos, 0:4] = freqs[col]                  # A,C,G,U
        feats[pos, 4] = gap_rate[col].astype(np.float32)
        feats[pos, 5] = conservation[col]
        pos += 1

    # if mapping didn’t fill all positions, still return (zeros for remainder)
    return feats

In [ ]:
train_seq = pd.read_csv(os.path.join(DATA_DIR, "train_sequences.csv"))
train_seq["seq_len"] = train_seq["sequence"].str.len()

# Filter to short sequences + MSA file exists
candidates = train_seq[train_seq["seq_len"] <= MAX_LEN].copy()
candidates["has_msa"] = candidates["target_id"].apply(lambda tid: os.path.exists(os.path.join(MSA_DIR, f"{tid}.MSA.fasta")))
candidates = candidates[candidates["has_msa"]].reset_index(drop=True)

print("Short+MSA candidates:", len(candidates))

# Pick first N_TARGETS (deterministic). You can also sample.
train_targets = candidates["target_id"].iloc[:N_TARGETS].tolist()
target_to_seq = dict(zip(candidates["target_id"], candidates["sequence"]))

print("Using targets:", len(train_targets))
print("Example target:", train_targets[0], "len:", len(target_to_seq[train_targets[0]]))

In [ ]:
LABELS_PATH = os.path.join(DATA_DIR, "train_labels.csv")

usecols = ["ID", "resid", "x_1", "y_1", "z_1"]
targets_set = set(train_targets)

# store coords per target
coords_map = {tid: None for tid in train_targets}

# initialize arrays
for tid in train_targets:
    L = len(target_to_seq[tid])
    coords_map[tid] = np.full((L, 3), np.nan, dtype=np.float32)

chunk_size = 500_000
kept = 0

for chunk in pd.read_csv(LABELS_PATH, usecols=usecols, chunksize=chunk_size, low_memory=False):
    # parse target_id from ID like "8ZNQ_123"
    tid = chunk["ID"].str.split("_").str[0]
    chunk = chunk.assign(target_id=tid)

    chunk = chunk[chunk["target_id"].isin(targets_set)]
    if chunk.empty:
        continue

    # fill into coords_map
    for row in chunk.itertuples(index=False):
        t = row.target_id
        r = int(row.resid) - 1  # resid is 1-based
        if 0 <= r < coords_map[t].shape[0]:
            coords_map[t][r, 0] = row.x_1
            coords_map[t][r, 1] = row.y_1
            coords_map[t][r, 2] = row.z_1
            kept += 1

print("Filled label points:", kept)

# quick sanity
some = train_targets[0]
print("Example coords finite:", np.isfinite(coords_map[some]).all(axis=1).mean(), "target:", some)

In [ ]:
base_to_id = {"A":0, "C":1, "G":2, "U":3}
UNK = 4

def encode_tokens(seq):
    return np.array([base_to_id.get(ch, UNK) for ch in seq], dtype=np.int32)

X_tok = np.full((len(train_targets), MAX_LEN), UNK, dtype=np.int32)
X_msa = np.zeros((len(train_targets), MAX_LEN, 6), dtype=np.float32)
Y_dlt = np.zeros((len(train_targets), MAX_LEN, 3), dtype=np.float32)
Msk   = np.zeros((len(train_targets), MAX_LEN), dtype=np.float32)

bad_msa = 0
used = 0

for i, tid in enumerate(train_targets):
    seq = target_to_seq[tid]
    L = len(seq)
    coords = coords_map[tid]  # (L,3) with NaNs possible

    feats = msa_features_for_target(tid, seq)
    if feats is None:
        bad_msa += 1
        continue

    tok = encode_tokens(seq)

    # mask: residue has all finite coords
    finite = np.isfinite(coords).all(axis=1)
    if finite.sum() < 5:
        continue

    # build deltas: d[i] = coord[i]-coord[i-1], d[0]=0
    dlt = np.zeros((L,3), dtype=np.float32)
    dlt[1:] = coords[1:] - coords[:-1]

    # only trust deltas where both i and i-1 coords are finite
    good_delta = finite.copy()
    good_delta[1:] = finite[1:] & finite[:-1]
    good_delta[0] = False  # first delta not informative

    X_tok[i, :L] = tok
    X_msa[i, :L, :] = feats
    Y_dlt[i, :L, :] = np.nan_to_num(dlt, nan=0.0).astype(np.float32)
    Msk[i, :L] = good_delta.astype(np.float32)

    used += 1

print("Targets with usable MSA+labels:", used, "/", len(train_targets), "bad_msa:", bad_msa)

# keep only used rows
keep_idx = np.where(Msk.sum(axis=1) > 0)[0]
X_tok = X_tok[keep_idx]
X_msa = X_msa[keep_idx]
Y_dlt = Y_dlt[keep_idx]
Msk   = Msk[keep_idx]

print("Final shapes:", X_tok.shape, X_msa.shape, Y_dlt.shape, Msk.shape)
print("Total supervised deltas:", float(Msk.sum()))

In [ ]:
ds = tf.data.Dataset.from_tensor_slices((X_tok, X_msa, Y_dlt, Msk))
ds = ds.shuffle(2048, seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)

tok_in = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32)
msa_in = tf.keras.Input(shape=(MAX_LEN, 6), dtype=tf.float32)

emb = tf.keras.layers.Embedding(5, 32)(tok_in)  # (B,L,32)
x = tf.keras.layers.Concatenate()([emb, msa_in])  # (B,L,38)

x = tf.keras.layers.Conv1D(128, 7, padding="same", activation="relu")(x)
x = tf.keras.layers.Dropout(0.1)(x)
x = tf.keras.layers.Conv1D(128, 7, padding="same", activation="relu")(x)
x = tf.keras.layers.Dropout(0.1)(x)
x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)

dlt_pred = tf.keras.layers.Dense(3)(x)  # (B,L,3)
model = tf.keras.Model([tok_in, msa_in], dlt_pred)

def masked_mse(y_true, y_pred, mask):
    mask3 = tf.expand_dims(mask, -1)
    se = tf.square(y_true - y_pred) * mask3
    return tf.reduce_sum(se) / (tf.reduce_sum(mask3) + 1e-6)

opt = tf.keras.optimizers.Adam(1e-3)

@tf.function
def train_step(toks, msas, y, m):
    with tf.GradientTape() as tape:
        yp = model([toks, msas], training=True)
        loss = masked_mse(y, yp, m)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss

for epoch in range(EPOCHS):
    losses = []
    for toks, msas, y, m in ds:
        losses.append(train_step(toks, msas, y, m).numpy())
    print(f"epoch {epoch+1}/{EPOCHS}: delta_loss={float(np.mean(losses)):.6f}")

In [ ]:
test_seq = pd.read_csv(os.path.join(DATA_DIR, "test_sequences.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
sub = sample_sub.copy()

coord_cols = [c for c in sub.columns if c.startswith(("x_", "y_", "z_"))]
sub[coord_cols] = sub[coord_cols].astype(np.float32)

def smooth_noise(L, scale=0.2, win=25):
    # Low-frequency noise by moving average smoothing, safe for small L
    n = rng.normal(0.0, scale, size=(L, 3)).astype(np.float32)
    if L <= 2:
        return n

    # Ensure window is odd and <= L
    win = int(win)
    win = min(win, L)
    if win % 2 == 0:
        win -= 1
    if win <= 1:
        return n

    k = (np.ones(win, dtype=np.float32) / win)

    out = np.zeros_like(n)
    for d in range(3):
        y = np.convolve(n[:, d], k, mode="same")

        # Safety: if convolve returns wrong length, center-crop or pad
        if len(y) > L:
            start = (len(y) - L) // 2
            y = y[start:start+L]
        elif len(y) < L:
            y = np.pad(y, (0, L - len(y)), mode="edge")

        out[:, d] = y.astype(np.float32)

    return out

pred5 = {}  # target_id -> (5,L,3)

for _, row in test_seq.iterrows():
    tid = row["target_id"]
    seq = row["sequence"]
    L_full = len(seq)

    tok = encode_tokens(seq)
    feats = msa_features_for_target(tid, seq)
    if feats is None:
        feats = np.zeros((L_full, 6), dtype=np.float32)

    # truncate if needed
    L = min(L_full, MAX_LEN)
    x_tok = np.full((1, MAX_LEN), UNK, dtype=np.int32)
    x_msa = np.zeros((1, MAX_LEN, 6), dtype=np.float32)
    x_tok[0, :L] = tok[:L]
    x_msa[0, :L, :] = feats[:L, :]

    dlt = model.predict([x_tok, x_msa], verbose=0)[0]  # (MAX_LEN,3)
    dlt = dlt[:L].astype(np.float32)

    # reconstruct coords from deltas (start at origin)
    coords = np.zeros((L,3), dtype=np.float32)
    coords[1:] = np.cumsum(dlt[1:], axis=0)

    # expand to full length (zeros beyond MAX_LEN)
    base = np.zeros((L_full,3), dtype=np.float32)
    base[:L] = coords

    # 5 candidates: base + smooth perturbations
    arr = np.zeros((5, L_full, 3), dtype=np.float32)
    arr[0] = base
    for k in range(1, 5):
        noise = smooth_noise(L_full, scale=0.3, win=31)
        arr[k] = base + noise
    pred5[tid] = arr

print("Built predictions for test targets:", len(pred5))

# Fill submission
tid_res = sub["ID"].str.rsplit("_", n=1, expand=True)
sub["_target_id"] = tid_res[0]
sub["_resid"] = tid_res[1].astype(int)

for k in [1,2,3,4,5]:
    sub[f"x_{k}"] = 0.0
    sub[f"y_{k}"] = 0.0
    sub[f"z_{k}"] = 0.0

for tid, g in sub.groupby("_target_id", sort=False):
    arr = pred5.get(tid)
    if arr is None:
        continue
    idx = g["_resid"].to_numpy() - 1
    L = arr.shape[1]
    ok = (idx >= 0) & (idx < L)
    for k in range(5):
        xyz = np.zeros((len(idx), 3), dtype=np.float32)
        xyz[ok] = arr[k, idx[ok], :]
        sub.loc[g.index, [f"x_{k+1}", f"y_{k+1}", f"z_{k+1}"]] = xyz

sub.drop(columns=["_target_id","_resid"], inplace=True)

zeros = (sub[["x_1","y_1","z_1"]].to_numpy() == 0).all(axis=1).sum()
print("Rows with x_1,y_1,z_1 all zero:", int(zeros), "out of", len(sub))

sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv")
sub.head()
